ST554_Final Project   
Author: Joy Zhou  
Date: 4/21/2026

# Introduction   
This project is designed to assess the ability to use Apache Spark for processing streaming data and for training machine learning models in a scalable computing environment.


We will use a modified dataset from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city) on power consumption in Tetouan City, the project focuses on modeling the relationship between electricity consumption across different city zones and key influencing factors such as time of day, temperature, and humidity.   


By leveraging Spark's data processing and machine learning libraries, a predictive model will be developed using historical data and then applied to incoming data streams to perform real-time monitoring and prediction of power consumption. This project integrates concepts from big data processing, machine learning, and streaming analytics, highlighting the practical application of Spark in handling real-world, data-intensive problems.


# Part 1 Fitting the Model



The dataset power_ml_data.csv is available at https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv.
The data should first be loaded into a pandas DataFrame using the `pd.read_csv()` function and then converted into a Spark DataFrame. In this analysis, `Power_Zone_3` is treated as the response variable, while all remaining variables are used as predictors.

## Read in data

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 10:34:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/26 10:34:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


- First we read `power_ml_data` into a standard pandas data frame named `power_data` using the pd.read_csv() function.


In [2]:
power_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


- Convert `power_data` to a spark data frame `power`

In [3]:
#Convert this to a spark data frame
power = spark.createDataFrame(power_data)
power.show(5)
#We are going to treat the Power_Zone_3 variable as our response variable

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## Data transformation
We will fit an elastic net regression using cross-validation (CV), without performing an explicit training-test split. Instead, model selection and performance assessment are conducted entirely through cross-validation on the available dataset.   

Feature transformations are performed using Spark MLlib utilities, and the transformed variables are assembled into a machine learning pipeline to ensure a consistent and reproducible workflow.

- let's check the varaible types by inspectiong the dataset schema.


In [4]:
power.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



- Inspection of the schema shows that the Hour column is defined as a LongType. We therefore apply an SQLTransformer to cast this variable to DoubleType.

In [5]:
from pyspark.ml.feature import SQLTransformer

sqlTrans = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__")

To inspect the results of the SQL transformation, we apply sqlTrans to the power dataset and displayed a sample of the transformed records.

In [6]:
#inspect of records of sqlTrans
transformed_power = sqlTrans.transform(power)
transformed_power.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335

- We apply a binarization step to the Hour variable using a cutoff of 6.5, creating a new binary feature, night_vs_day, to indicate nighttime (Hour < 6.5) versus daytime (Hour ≥ 6.5) for downstream modeling.
    

In [7]:
from pyspark.ml.feature import Binarizer

binaryTrans = Binarizer(threshold=6.5, inputCol="Hour_double", outputCol="night_vs_day")

#inspect the transformer results
binary = binaryTrans.transform(
    sqlTrans.transform(power))
binary.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0

- Although the Month variable is stored as a LongType, it represents a categorical feature rather than a continuous numeric value. Therefore, we apply one-hot encoding to the `Month` column to generate binary indicator variables, allowing the model to capture seasonal effects without assuming an ordinal or linear relationship among months.

In [8]:
from pyspark.ml.feature import OneHotEncoder, VectorAssembler

# Location one-hot encoding
month_encoder = OneHotEncoder(inputCol="Month", outputCol="Month_ind", dropLast=True)

We fit the month_encoder model to inspect the transformed records to ensure that the one-hot encoding is performed correctly.

In [9]:
#inspect the one-hot encode results
model = month_encoder.fit(binary)
encoded = model.transform(binary)
encoded.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|     Month_ind|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|       


- Principal Component Analysis (PCA) is used to reduce the dimensionality of the environmental variables Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows. The variables are assembled into a feature vector and standardized using [StandardScaler](https://www.sparkcodehub.com/pyspark/mllib/standard-scaler) to mitigate the influence of variables with larger magnitudes. PCA is then applied to extract two uncorrelated principal components, which provide lower‑dimensional representations for subsequent analysis and modeling.


In [10]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml import Pipeline

#assemble raw features
assembler = VectorAssembler(inputCols=["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol="pcaFeatures")

# Scale features (mean=0, std=1)
scaler = StandardScaler(inputCol="pcaFeatures", outputCol="scaledFeatures", withMean=True, withStd=True)

#PCA on scaled features
pca = PCA( k=2, inputCol="scaledFeatures", outputCol="pcaOutput")

#inspect the results
#set up pipeline
pipeline = Pipeline(stages=[assembler, scaler, pca])

# Fit and transform
pca_model = pipeline.fit(encoded)
pca_transformed = pca_model.transform(encoded)

pca_transformed.select("scaledFeatures", "pcaOutput").show(5, truncate=False)

+----------------------------------------------------------------------------------------------------+---------------------------------------+
|scaledFeatures                                                                                      |pcaOutput                              |
+----------------------------------------------------------------------------------------------------+---------------------------------------+
|[-2.1079477438809433,0.3542085264292604,-0.7996341497278044,-0.6900839531060153,-0.6025312519793238]|[2.069074321346372,0.8078678829472266] |
|[-2.1328903699941857,0.3991947174962608,-0.7996341497278044,-0.6900121009460847,-0.6028048802947571]|[2.1029210638806535,0.8152476222806394]|
|[-2.1502641992178924,0.3991947174962608,-0.800911098051248,-0.690042354487108,-0.6026841619203012]  |[2.112066263379101,0.8227151924829961] |
|[-2.183291676554047,0.4313277111155467,-0.7996341497278044,-0.6899326854008982,-0.6027163534868227] |[2.1435031847422197,0.8329135817940967]|

As the output above, we combined several environmental measurements into a single dataset and used PCA to compress them into two summary variables that capture most of the information. These summaries were then used instead of the original variables in later analyses.

- we will rename our response variable `Power_Zone_3` as label

In [11]:
sqlLabel = SQLTransformer(
    statement = """
                SELECT *, Power_Zone_3 AS label FROM __THIS__
                """
)

- Use `VectorAssembler()` to put all predictors into a single features vector, which includes:   
    - two fitted PCA features (pcaOutput)      
    - a binary Hour variable (night_vs_day)   
    - Power_Zone_1   
    - Power_Zone_2   
    - Month indicator variables (Month_ind)   
Since the PCA features were standardized in previous steps, the additional non‑PCA features will also be standardized to ensure consistency across all predictors.

In [12]:
#assemble assembleFeatures
features_assembler = VectorAssembler(
    inputCols=["pcaOutput", "night_vs_day", "Power_Zone_1", "Power_Zone_2", "Month_ind"],
    outputCol="assembledFeatures"
)

#scale again after adding non‑PCA features.
final_scaler = StandardScaler(
    inputCol="assembledFeatures",
    outputCol="features",
    withMean=True,
    withStd=True
)

assembled_df = features_assembler.transform(pca_transformed)
scaler_model = final_scaler.fit(assembled_df)
featurestrans = scaler_model.transform(assembled_df)

# Inspect assembled features
featurestrans.select("assembledFeatures", "features").show(5, truncate=False)

+------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|assembledFeatures                                                                   |features                                                                                                                                                                                                                                                                                                                              |
+------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------

From the output above, we confirmed that the transformation pipeline was completed and the the dataset is ready for modeling.

## Fit an Elastic Net Model
We next fit an Elastic Net regression model using the `CrossValidator()` and `LinearRegression()` function. Model hyperparameters are selected via cross-validation over a predefined grid consisting of all combinations of the following values:   

- regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

We are now ready to set up the pipeline, which includes transcormations and the model to be fitted. We use the `Pipeline()` function from the `pyspark.ml` module to set up a squential workflow of transformations/estimators.

In [13]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline

# Elastic Net regression
lr = LinearRegression(featuresCol='features', labelCol='label') #create a liner regression instance

#define parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

- Set up the pipeline 

In [14]:
pipeline = Pipeline(stages=[
    sqlTrans,              # Cast hour 
    binaryTrans,           # night_vs_day
    month_encoder,         # Month one-hot encoding
    sqlLabel,              # Rename response variable
    assembler,             # Assemble environmental vars to create pcaFeatures
    scaler,                # Scale pcaFeatures
    pca,                   # PCA produces pcaOutput
    features_assembler,    # Combine pcaOutput + indicators to get assembledFeatures
    final_scaler,          # Scale final feature vector to features
    lr                     # Elastic Net regression model
])

We apply `CrossValidator()` to carry out 5-fold cross-validation over the specified hyperparameter grid. Model performance is evaluated using `RegressionEvaluator()`, with RMSE specified via the metricName argument. This procedure will split the dataset into five folds. In each iteration, the model is trained on four folds and validated on the remaining fold for every hyperparameter combination. The performance metric (rmse) is then averaged across all five folds, and the model with the lowest average rmse is selected as the best model.

In [15]:
from pyspark.ml.evaluation import RegressionEvaluator
#initialize cross-validator
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    ),
    numFolds=5,    #5-fold cross-validation
    seed = 42
)
#fitting the model, and choose the best set of parameters
cv_model = cv.fit(power)

26/04/26 10:35:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/26 10:35:46 WARN Instrumentation: [111ac130] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 10:35:47 WARN Instrumentation: [111ac130] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/26 10:35:49 WARN Instrumentation: [dac5bce6] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 10:35:49 WARN Instrumentation: [dac5bce6] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/26 10:35:51 WARN Instrumentation: [e1a3c5a2] regParam is zero, which might cause numerical instability and overfitting.
26/04/26 10:35:51 WARN Instrumentation: [e1a3c5a2] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Evaluating the best model
- We will use the bestModel attribute to retrive the best model from cross validation
- Then, extract the final stage of the pipeline, the trained LinearRegression model, from the selected best model for further evaluation and interpretation.

In [16]:
# Retrieves the best pipeline model from the cross-validation
best_model = cv_model.bestModel
#Extracts the last stage of the pipeline
best_lr = best_model.stages[-1]  # LinearRegression is last stage

Report the optimal values of the tuning parameters (regParam and elasticNetParam) selected by cross‑validation using `getRegParam()` and `getElasticNetParam()` methods.

In [17]:
# Printing parameters for verification
print(f"Best regParam: {best_lr.getRegParam()}")
print(f"Best elasticNetParam: {best_lr.getElasticNetParam()}")

Best regParam: 0.05
Best elasticNetParam: 0.05


Both parameters were selected as 0.05 because cross‑validation identified a model with mild, Ridge‑dominant regularization that best balances bias and variance for the standardized feature set.

Report the CV error   
- The cross‑validation error is computed as the average RMSE across the five folds, evaluated over all hyperparameter combinations.

In [18]:
#report CV error
avg_rmse = min(cv_model.avgMetrics)
print(f"Best CV RMSE: {avg_rmse:.4f}")

Best CV RMSE: 2174.9962


Report training set RMSE
- The fitted regression model with the best parameters now has a transform method that can be used to make predictions using the entire dataset. The training set RMSE is obtained by applying the best fitted pipeline model to the entire dataset using the transform() method and evaluating prediction error using RMSE.

In [19]:
#apply the best fitted pipeline model to the entire dataset
train_predictions = best_model.transform(power)

#define evaluator
evaluator=RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    )
# Evaluate RMSE
train_rmse = evaluator.evaluate(train_predictions)
print(f"Training RMSE: {train_rmse:.4f}")

Training RMSE: 2174.4490


The model outputs are used to generate predictions, from which a residual column is computed as the difference between the observed label and the predicted value. The `.withColumn()` method is used to create this residual. A resulting data frame is then displayed containing the residuals, the label column, and the model predictions.

In [20]:
from pyspark.sql.functions import col

residuals_df = train_predictions.withColumn("residual", col("label") - col("prediction"))

residuals_df.select( "label", "prediction", "residual",).show(10, truncate=False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20936.226841006064|-695.2629810060644|
|20131.08434|18701.586468095124|1429.4978719048777|
|19668.43373|18237.18946977339 |1431.2442602266092|
|18899.27711|17615.89221625058 |1283.3848937494186|
|18442.40964|17017.381842683895|1425.0277973161064|
|18130.12048|16540.614302205628|1589.5061777943738|
|17945.06024|16114.684233894432|1830.3760061055673|
|17459.27711|15742.038334202176|1717.238775797823 |
|17025.54217|15282.55797300917 |1742.9841969908302|
|16794.21687|14934.899321422648|1859.3175485773518|
+-----------+------------------+------------------+
only showing top 10 rows


## Conclusion
Using the Power dataset, an elastic net linear regression model was fitted through a pipeline and tuned via 5‑fold cross‑validation, with RMSE serving as the evaluation metric. The optimal regularization parameters were selected based on cross‑validated performance. The best‑fitted pipeline model was then applied to the entire dataset to compute the training set RMSE, which was 2174.449. Prediction residuals were subsequently calculated as the difference between the observed and predicted values.
During this process, transformations were applied to the hour and month variables, and all features were standardized prior to PCA and model fitting to mitigate the influence of variables with larger magnitudes. The use of a pipeline enabled efficient tracking of feature transformations and streamlined the modeling workflow, while Spark facilitated scalable and efficient model fitting. In the next section, methods for handling streaming data will be explored.

# Streaming Part


## Reading a stream
We set up a Structured Streaming read operation that monitors a specified folder for incoming CSV files, which created by .py file).

In [72]:
#create spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

- The schema from the power DataFrame is extracted and reused to ensure consistent column types when setting up the streaming CSV source. The header option is set to True so that all incoming files are correctly read with column headers.

In [73]:
#set up schema
power_schema = power.schema

#set up the stream df
stream_df = spark.readStream.schema(power_schema).option("header", True).csv("csv_files")

## Transform/Aggregation Step
The streaming dataset is processed using two separate transformations derived from the same input stream, following these steps:

- The first transformation applies the fitted pipeline model to generate prediction values, residuals are computed as the difference between the observed values and the model predictions.

In [74]:
from pyspark.sql.functions import col

# Apply the fitted feature-engineering pipeline and the model to streaming data
predictions_df = (
    best_model
        .transform(stream_df)
        .withColumn("residual", col("label") - col("prediction"))     #Create a residual column
        .select("label", "prediction", "residual")
)

- The second transformation operates on the original stream and modifies the response variable `Power_Zone_3` to be named label.

In [75]:
#2nd treansformation
label_df = stream_df.withColumn("label", col("Power_Zone_3")).drop("Power_Zone_3")

- The two transformed streams are then joined on the common label column using `join()` method.

In [76]:
joined_stream = predictions_df.join(label_df, on="label")

## Write the stream

The `.writeStream` method is used to output the transformed streaming data to the console in append mode. The `.start()` method initiates the streaming query and begins monitoring the input directory for new data. The five CSV files are added to the empty `csv_files` folder one at a time to be processed incrementally.

In [77]:
#Write the transformed streaming data to the console and start the streaming query
query = (
    joined_stream
        .writeStream
        .format("console") 
        .outputMode("append")
        .start()
)

26/04/26 13:17:02 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-04bb5ac0-5beb-49a4-8055-53d7ad7df561. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/26 13:17:02 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14460.25105| 21130.44852431516|  -6670.19747431516|      24.01|    83.0|     4.915|                0.424|        0.389| 22306.44518| 15026.58228|    7|   5|
|    18176.0|17787.955390225306|  388.0446097746935|      20.02|   51.92|     0.083|                404.0|        383.4| 32972.74489| 18593.89002|    4|  17|
|18020.40486| 17596.67362182164|  423.7312381783595|      21.85|   58.09|     0.071|                843.0|       

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|10259.27711| 9950.143263356485|  309.1338466435154|       11.4|    80.3|     4.919|                297.2|        45.16| 25624.61538| 17959.09091|   11|  10|
|7341.176471|7372.5272941189705|-31.350823118970766|      11.67|    79.3|     0.086|                 88.4|         22.0| 24693.53612| 18833.99816|   12|   9|
|11178.79518|  9642.92871739906| 1535.8664626009395|      4.509|    74.5|     0.084|                6.643|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25891.08434|24908.851766921212|  982.2325730787888|      14.99|    74.8|     0.071|                 0.07|        0.115| 42027.34177| 25469.90881|    1|  19|
|14869.78723| 13332.87717394178| 1536.9100560582192|      18.61|    84.2|     0.085|                0.069|        0.089| 33973.91685| 20404.97925|   10|  23|
|25266.95925|26490.063879534322|-1223.1046295343222|      25.49|   55.92|      4.92|                708.0|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|27432.72727|26653.788821352056|   778.938448647943|       18.4|    73.0|     0.078|                3.077|        2.836| 43568.91281| 23352.34216|    4|  20|
|15398.70968|16480.881988636145|-1082.1723086361453|      20.71|   61.77|     4.916|                297.0|        322.1| 31796.42553| 19979.26829|    3|  17|
|9323.409364|  9427.61415383561|-104.20478983561043|       8.02|    85.3|     0.089|                286.2|       

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12231.97568|13649.709898703368|-1417.7342187033682|      20.66|    82.3|     4.915|                224.2|        171.3| 33885.68928| 25674.27386|   10|  13|
|11125.16129|12748.515892268264| -1623.354602268264|      13.36|    86.4|     0.083|                0.029|        0.078| 21986.04255| 13693.90244|    3|   6|
|    13920.0| 13747.23747399182| 172.76252600818043|      13.55|    84.8|     0.075|                0.062|      

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|12319.51368|14711.804802973717|-2392.2911229737165|      23.36|   67.11|     4.921|                184.1|        158.0| 36097.68053| 23392.53112|   10|  14|
|12549.39759|14166.951853011344|-1617.5542630113432|      5.814|    83.1|     0.085|                13.12|        11.54| 26053.67089|  17576.8997|    1|   8|
|28683.63636|27600.994477622407|  1082.641882377593|      18.17|    73.0|     0.079|                11.67|      

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14385.52764|13560.923973830857|  824.6036661691433|      13.28|   66.66|     0.084|                0.051|        0.115|  22924.0678| 13706.99088|    2|   5|
| 20068.8049|18212.390622708852| 1856.4142772911473|      21.17|    78.9|     0.281|                0.077|          0.1|  38950.0885| 22887.31809|    9|  23|
|10967.27273|10100.157164303255|  867.1155656967458|      13.35|    88.4|     0.085|                146.9|      

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|13873.42186| 13514.10972140815| 359.3121385918512|      21.46|   56.55|     0.271|                0.095|        0.093|  29347.9646| 17259.04366|    9|   1|
|30553.30544|27660.550047202756| 2892.755392797244|      27.95|    71.8|     4.916|                308.0|        237.5| 36760.66445| 22682.27848|    7|  14|
|10403.85542|12258.108067431049|-1854.252647431049|      20.38|    72.8|     0.072|                168.2|        139.

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|25216.64322|22546.741182708516| 2669.9020372914856|       8.39|   54.42|     0.085|                0.062|        0.108| 39459.66102| 22964.13374|    2|  20|
|  12436.231| 14083.02315641322|-1646.7921564132193|      20.76|    84.8|     0.067|                27.31|        23.58| 34881.40044|  22847.3029|   10|  15|
|14227.13085|14968.607435272399| -741.4765852723995|      17.47|   66.01|     0.085|                16.06|      

When We start the query, We read in streaming datasets using the method that created in `stream.py` file. We will submit `stream.py' in a python console.


In [78]:
# Stop the streaming query after processing all input files
query.stop()

26/04/26 13:25:06 WARN DAGScheduler: Failed to cancel job group 53d6b42e-9f87-4dc2-aef3-1d03a45e6e5f. Cannot find active jobs for it.
26/04/26 13:25:07 WARN DAGScheduler: Failed to cancel job group 53d6b42e-9f87-4dc2-aef3-1d03a45e6e5f. Cannot find active jobs for it.


In [32]:
#Write the transformed streaming data to the console and start the streaming query
from pyspark.sql.functions import col

max_batches = 20

def show_each_batch(batch_df, batch_id):
    if batch_id < max_batches:
        print(f"\n========== BATCH {batch_id} ==========")
        batch_df.show(truncate=False)
    else:
        spark.streams.active[0].stop()

query = (
    joined_stream
        .writeStream
        .foreachBatch(show_each_batch)
        .format("console") 
        .outputMode("append")
        .start()
)

26/04/26 11:33:59 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-2707ff6a-fd43-49a5-86ad-bb049e768750. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/26 11:33:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
|9018.007203| 7384.777133840995| 1633.2300691590044|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|10712.12485|13222.033506653604|-2509.9086566536043|
|16314.21687| 17759.40134843298|-1445.1844784329805|
|15294.19355|13812.500261585516| 1481.6932884144844|
+-----------+------------------+-------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|10712.12485|13222.033506653604|-2509.9086566536043|
|10712.12485|13222.033506653604|-2509.9086566536043|
|10712.12485|13222.033506653604|-2509.9086566536043|
|16314.21687| 17759.40134843298|-1445.1844784329805|
|16314.21687| 17759.40134843298|-1445.1844784329805|
|16314.21687| 17759.40134843298|-1445.1844784329805|
|15294.19355|13812.500261585516| 1481.6932884144844|
|15294.19355|13812.500261585516| 1481.6932884144844|
|1

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|10712.12485|13222.033506653604|-2509.9086566536043|
|10712.12485|13222.033506653604|-2509.9086566536043|
|10712.12485|13222.033506653604|-2509.9086566536043|
|10712.12485|13222.033506653604|-2509.9086566536043|
|1

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+
|      label|        prediction|           residual|
+-----------+------------------+-------------------+
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|9018.007203| 7384.777133840995| 1633.2300691590044|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|8356.960486| 7148.773780703965| 1208.1867052960351|
|1

...

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+-----------------+------------------+
|      label|       prediction|          residual|
+-----------+-----------------+------------------+
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+-----------------+------------------+
|      label|       prediction|          residual|
+-----------+-----------------+------------------+
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+-----------------+------------------+
|      label|       prediction|          residual|
+-----------+-----------------+------------------+
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+-----------------+------------------+
|      label|       prediction|          residual|
+-----------+-----------------+------------------+
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+-----------------+------------------+
|      label|       prediction|          residual|
+-----------+-----------------+------------------+
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633.2300691590044|
|9018.007203|7384.777133840995|1633

In [ ]:
query = (
         final_stream_output
        .writeStream
        .format("console")
        .outputMode("append")
        .start()
)

In [31]:
# Stop the streaming query after processing all input files
query.stop()

26/04/26 11:30:12 WARN DAGScheduler: Failed to cancel job group 8c4fcbf2-579d-49c6-a009-7a2e7402f6f1. Cannot find active jobs for it.
26/04/26 11:30:13 WARN DAGScheduler: Failed to cancel job group 8c4fcbf2-579d-49c6-a009-7a2e7402f6f1. Cannot find active jobs for it.
